In [ ]:
# HuggingFace Authentication

from huggingface_hub import login

# Paste your HuggingFace access token here
# Get token from: https://huggingface.co/settings/tokens
login("HF_TOKEN") # my token removed for security purposes.

c:\Users\kirut\anaconda3\envs\hf_assign\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load FLAN-T5 Small Model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-small"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Optional: reduce CPU usage
torch.set_num_threads(4)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 268.06it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded successfully!


In [ ]:
# Response generation function

def generate_response(prompt):

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        min_new_tokens=50,
        do_sample=False,
        temperature = 0.7,
        top_p = 0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
        
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Test model
prompt = """
You are a helpful AI tutor.

Write a clear explanation in 4–5 sentences.
Explain step-by-step in simple words.
Do NOT answer in one word.
Give a technical definition.
Do not guess meanings.
Do not talk about languages or etymology.

Question: What is Generative AI?

Detailed Answer:
"""

print(generate_response(prompt))

The generative AI is the process of learning and analyzing information. It is a tool that helps you learn to understand and analyze information. The encoding of information is based on the data of the person who is using it. The data is used to identify the people who are using it and how they are doing it.


In [39]:
# LangChain Integration 

from langchain_core.language_models.llms import LLM

class LocalHFLLM(LLM):

    def _call(self, prompt, stop=None):

        #  Force structured instruction
        formatted_prompt = f"""
You are a helpful AI tutor.

Context:
LangChain is a software framework used to build applications powered by large language models (LLMs).
It helps connect LLMs with external data, tools, and workflows.

Task:
Explain the topic below clearly in 4 simple sentences using the context.

Question: {prompt}

Answer:
"""

        return generate_response(formatted_prompt)

    @property
    def _llm_type(self):
        return "huggingface_local"


# Create LangChain LLM
llm = LocalHFLLM()

print(llm.invoke("Explain LangChain in simple terms."))

LangChain: a software framework used to build applications powered by large language models (LLMs). It helps connect LLMs with external data, tools, and workflows. 4 simple sentences in sentence 1. Explain LangChalin:


In [ ]:

# PromptTemplate + Chain (NEW STYLE)


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create prompt template
prompt_template = PromptTemplate.from_template(
"""
Explain the following topic clearly in 4 sentences.

Topic: {topic}

Explanation:
"""
)

# Create chain using Runnable pipeline
chain = prompt_template | llm | StrOutputParser()

# Run chain
result = chain.invoke({"topic": "Large Language Models"})

print(result)


Language models are used to build applications. They can be built by a large number of languages. They are used by many other languages. It is a good way to learn languages. The language model is based on the language model. It can be used to create and build applications for a variety of languages, including languages.


In [ ]:
# Chat Prompt Template (NEW LANGCHAIN)

from langchain_core.prompts import ChatPromptTemplate

# System + Human messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor."),
    ("human", "Explain {concept} with an example.")
])

# Format prompt
formatted_prompt = chat_prompt.invoke(
    {"concept": "LangChain"}
)

# Convert to string (needed for our HF model)
final_prompt = formatted_prompt.to_string()

# Run model
print(llm.invoke(final_prompt))

LangChain: a software framework used to build applications powered by large language models (LLMs). It helps connect LLMs with external data, tools, and workflows. Task: Explain LangChalin with an example.


In [ ]:
# Compare Normal vs Chat Prompt

print("---- Normal Prompt ----")
print(llm.invoke("Explain LangChain briefly."))

print("\n---- Chat Prompt ----")
print(llm.invoke(final_prompt))

---- Normal Prompt ----
LangChain: Using LLMs to build applications. (Laughter) The LangChuite is a software framework used to build apps powered by large language models (LLMs). It helps connect LLM with external data, tools, and workflows. (Applause)

---- Chat Prompt ----
LangChain: a software framework used to build applications powered by large language models (LLMs). It helps connect LLMs with external data, tools, and workflows. Task: Explain LangChalin with an example.


Observations

HuggingFace models can be successfully integrated with LangChain to generate responses locally without external APIs.

PromptTemplate helps create structured prompts and improves the consistency of model outputs.

ChatPromptTemplate enables system and user role separation, making responses more guided and conversational.

Small HuggingFace models require clear instructions and context to reduce incorrect or irrelevant answers.

LangChain chaining (prompt | llm) simplifies building LLM workflows and improves prompt management.